<a href="https://colab.research.google.com/github/SOTOGALIO/GestionDatosIa/blob/main/Actividad_2_2_2_GestionIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cargar datos en colab

In [1]:
import pandas as pd

df = pd.read_csv("data.csv")

df.head()


,UserID,Age,Gender,VRHeadset,Duration,MotionSickness,ImmersionLevel
0,1,40,Male,HTC Vive,13.598508,8,5
1,2,43,Female,HTC Vive,19.950815,2,2
2,3,27,Male,PlayStation VR,16.543387,4,2
3,4,33,Male,HTC Vive,42.574083,6,3
4,5,51,Male,PlayStation VR,22.452647,4,2


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UserID          1000 non-null   int64  
 1   Age             1000 non-null   int64  
 2   Gender          1000 non-null   object 
 3   VRHeadset       1000 non-null   object 
 4   Duration        1000 non-null   float64
 5   MotionSickness  1000 non-null   int64  
 6   ImmersionLevel  1000 non-null   int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 54.8+ KB


# Limpieza basica

## Eliminar nulos

In [3]:
df.isnull().sum()
df = df.dropna()

## Eliminar duplicados.

In [4]:
df = df.drop_duplicates()

## Validar rangos (MUY IMPORTANTE en este dataset)

In [5]:
# Edad razonable
df = df[(df["Age"] >= 10) & (df["Age"] <= 100)]

# Duración positiva
df = df[df["Duration"] > 0]

# Escalas válidas
df = df[(df["MotionSickness"] >= 1) & (df["MotionSickness"] <= 10)]
df = df[(df["ImmersionLevel"] >= 1) & (df["ImmersionLevel"] <= 5)]

# Estandarización

## Texto

In [6]:
df["Gender"] = df["Gender"].str.lower().str.strip()
df["VRHeadset"] = df["VRHeadset"].str.lower().str.strip()

## Nombres de columnas (MUY BUENA PRÁCTICA)

In [7]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

## Tipos de datos

In [8]:
df["age"] = df["age"].astype(int)
df["duration"] = pd.to_numeric(df["duration"], errors="coerce")

# Transformaciones

## Crear nuevas columnas

In [9]:
# Clasificación de edad
df["age_group"] = pd.cut(df["age"],
                        bins=[0,18,30,50,100],
                        labels=["teen","young_adult","adult","senior"])

## Clasificar nivel de inmersión

In [10]:
def clasificar_inmersion(x):
    if x <= 2:
        return "baja"
    elif x == 3:
        return "media"
    else:
        return "alta"

df["immersion_category"] = df["immersionlevel"].apply(clasificar_inmersion)

# Modularidad

In [11]:
def limpiar_datos(df):
    df = df.dropna()
    df = df.drop_duplicates()
    return df

def estandarizar(df):
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df["gender"] = df["gender"].str.lower().str.strip()
    return df

# Guardar dataset limpio

In [12]:
import os

output_dir = "/content/data/processed"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "vr_clean.csv"), index=False)

## readme

In [13]:
readme_text = """
# Limpieza de Dataset - VR Experiences

## Descripción
Este proyecto realiza limpieza y transformación del dataset de experiencias de realidad virtual.

## Procesos realizados:
- Eliminación de valores nulos
- Eliminación de duplicados
- Estandarización de texto
- Conversión de tipos de datos
- Filtrado de valores fuera de rango
- Creación de variables derivadas (age_group, immersion_category)

## Output:
Dataset limpio guardado en /data/processed/
"""

with open("README.md", "w") as f:
    f.write(readme_text)

from google.colab import files
files.download("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# parte 2


In [14]:
import pandas as pd
import re
import unicodedata

# ════════════════════════════════════════════════════════════════════════════════
# 0. CARGA DE DATOS
# ════════════════════════════════════════════════════════════════════════════════
df = pd.read_csv("data.csv")
print("▶ Filas iniciales:", len(df))
print(df.info())
df.head()

# ════════════════════════════════════════════════════════════════════════════════
# 1. NOMBRES DE COLUMNAS — normalizar primero para usar siempre lowercase
# ════════════════════════════════════════════════════════════════════════════════
df.columns = df.columns.str.lower().str.replace(" ", "_")

# ════════════════════════════════════════════════════════════════════════════════
# 2. LIMPIEZA BÁSICA
# ════════════════════════════════════════════════════════════════════════════════

# 2.1 Nulos y duplicados
df = df.dropna()
df = df.drop_duplicates()

# 2.2 Validar rangos (columnas ya en lowercase)
df = df[(df["age"] >= 10) & (df["age"] <= 100)]
df = df[df["duration"] > 0]
df = df[(df["motionsickness"] >= 1) & (df["motionsickness"] <= 10)]
df = df[(df["immersionlevel"] >= 1) & (df["immersionlevel"] <= 5)]
print("▶ Filas tras limpieza básica:", len(df))

# ════════════════════════════════════════════════════════════════════════════════
# 3. ESTANDARIZACIÓN DE TIPOS
# ════════════════════════════════════════════════════════════════════════════════
df["age"]      = df["age"].astype(int)
df["duration"] = pd.to_numeric(df["duration"], errors="coerce")

# ════════════════════════════════════════════════════════════════════════════════
# 4. LIMPIEZA SEMÁNTICA DE STRINGS
# ════════════════════════════════════════════════════════════════════════════════

# 4.1 Normalización unicode, control chars y espacios
def normalizar_texto(val):
    """Repara encoding, normaliza unicode NFKC, elimina control chars y colapsa espacios."""
    if pd.isna(val):
        return None
    val = unicodedata.normalize("NFKC", str(val))
    val = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", val)
    val = re.sub(r"\s+", " ", val).strip().lower()
    return val

df["gender"]    = df["gender"].apply(normalizar_texto)
df["vrheadset"] = df["vrheadset"].apply(normalizar_texto)

# 4.2 Canonización con mapas de alias
GENDER_MAP = {
    "male":   "masculino",
    "female": "femenino",
    "other":  "otro",
}
HEADSET_MAP = {
    "htc vive":       "HTC Vive",
    "playstation vr": "PlayStation VR",
    "oculus rift":    "Oculus Rift",
}

df["gender_std"]  = df["gender"].map(GENDER_MAP)
df["headset_std"] = df["vrheadset"].map(HEADSET_MAP)

# ════════════════════════════════════════════════════════════════════════════════
# 5. TRANSFORMACIONES NUMÉRICAS
# ════════════════════════════════════════════════════════════════════════════════

# 5.1 Conversión de unidades de duración
df["duration_min"] = df["duration"].round(2)
df["duration_seg"] = (df["duration"] * 60).round(0).astype(int)

# 5.2 Normalización 0-1 de escalas ordinales
df["motionsickness_norm"] = ((df["motionsickness"] - 1) / 9).round(3)
df["immersionlevel_norm"] = ((df["immersionlevel"] - 1) / 4).round(3)

# 5.3 Bins de duración y tolerancia al movimiento
df["duracion_tipo"] = pd.cut(
    df["duration"],
    bins=[0, 15, 35, 60],
    labels=["corta", "media", "larga"]
)
df["tolerancia_movimiento"] = pd.cut(
    df["motionsickness"],
    bins=[0, 3, 6, 10],
    labels=["alta tolerancia", "tolerancia media", "sensible"]
)

# ════════════════════════════════════════════════════════════════════════════════
# 6. COLUMNAS DERIVADAS (FEATURES DE NEGOCIO)
# ════════════════════════════════════════════════════════════════════════════════

# 6.1 Score de experiencia VR compuesto
df["vr_experience_score"] = (
    (df["immersionlevel"] * 2)
    - (df["motionsickness"] * 1.5)
    + (df["duration_min"] / 10)
).round(2)

# 6.2 Flag sesión problemática: alto mareo + larga duración
df["sesion_problematica"] = (
    (df["motionsickness"] >= 7) & (df["duration_min"] > 30)
)

# 6.3 Ratio inmersión / mareo
df["immersion_sickness_ratio"] = (
    df["immersionlevel"] / df["motionsickness"]
).round(3)

# 6.4 Segmento de usuario
def segmentar_usuario(row):
    if row["immersionlevel"] >= 4 and row["motionsickness"] <= 3:
        return "power_user"
    elif row["motionsickness"] >= 7:
        return "sensible"
    elif row["duration_min"] < 15:
        return "casual"
    else:
        return "regular"

df["user_segment"] = df.apply(segmentar_usuario, axis=1)

# 6.5 Grupo etario recalculado y décadas
def calcular_age_group(age):
    if age < 18:   return "teen"
    elif age < 30: return "young_adult"
    elif age < 50: return "adult"
    else:          return "senior"

df["age_group"]     = df["age"].apply(calcular_age_group)
df["age_group_std"] = df["age_group"]
df["decada"]        = (df["age"] // 10 * 10).astype(str) + "s"

# 6.6 Categoría e inmersión con rank ordinal
def clasificar_inmersion(x):
    if x <= 2:   return "baja"
    elif x == 3: return "media"
    else:        return "alta"

IMMERSION_ORDER = {"baja": 1, "media": 2, "alta": 3}
df["immersion_category"] = df["immersionlevel"].apply(clasificar_inmersion)
df["immersion_rank"]     = df["immersion_category"].map(IMMERSION_ORDER)

# ════════════════════════════════════════════════════════════════════════════════
# 7. ENCODING PARA ML
# ════════════════════════════════════════════════════════════════════════════════

# 7.1 Encoding numérico ordinal
df["gender_enc"]  = df["gender_std"].map({"masculino": 0, "femenino": 1, "otro": 2})
df["headset_enc"] = df["headset_std"].astype("category").cat.codes

# 7.2 One-hot encoding de headset
dummies = pd.get_dummies(df["headset_std"], prefix="headset")
df = pd.concat([df, dummies], axis=1)

# ════════════════════════════════════════════════════════════════════════════════
# 8. MÓDULOS REUTILIZABLES
# ════════════════════════════════════════════════════════════════════════════════

def limpiar_datos(df):
    """Limpieza básica: nulos, duplicados y validación de rangos."""
    df = df.dropna()
    df = df.drop_duplicates()
    df = df[(df["age"] >= 10) & (df["age"] <= 100)]
    df = df[df["duration"] > 0]
    df = df[(df["motionsickness"] >= 1) & (df["motionsickness"] <= 10)]
    df = df[(df["immersionlevel"] >= 1) & (df["immersionlevel"] <= 5)]
    return df

def estandarizar(df):
    """Estandarización de nombres de columnas y limpieza semántica de texto."""
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df["gender"]    = df["gender"].apply(normalizar_texto)
    df["vrheadset"] = df["vrheadset"].apply(normalizar_texto)
    return df

# ════════════════════════════════════════════════════════════════════════════════
# 9. RESUMEN FINAL Y GUARDADO
# ════════════════════════════════════════════════════════════════════════════════
print("\n▶ Shape final:", df.shape)
print("\n▶ Columnas generadas:", df.columns.tolist())
print("\n▶ Muestra de columnas clave:")
print(df[[
    "age", "decada", "gender_std", "headset_std",
    "duration_min", "duracion_tipo",
    "motionsickness_norm", "immersionlevel_norm",
    "vr_experience_score", "user_segment",
    "sesion_problematica", "immersion_rank"
]].head(8).to_string())

df.to_csv("/content/data/processed/vr_clean.csv", index=False)
print("\n✓ Dataset guardado en /content/data/processed/vr_clean.csv")

▶ Filas iniciales: 1000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UserID          1000 non-null   int64  
 1   Age             1000 non-null   int64  
 2   Gender          1000 non-null   object 
 3   VRHeadset       1000 non-null   object 
 4   Duration        1000 non-null   float64
 5   MotionSickness  1000 non-null   int64  
 6   ImmersionLevel  1000 non-null   int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 54.8+ KB
None
▶ Filas tras limpieza básica: 1000

▶ Shape final: (1000, 29)

▶ Columnas generadas: ['userid', 'age', 'gender', 'vrheadset', 'duration', 'motionsickness', 'immersionlevel', 'gender_std', 'headset_std', 'duration_min', 'duration_seg', 'motionsickness_norm', 'immersionlevel_norm', 'duracion_tipo', 'tolerancia_movimiento', 'vr_experience_score', 'sesion_problematica', 'immersion_sickness_ratio', '

# Agregar carga a base de datos

In [16]:
import pandas as pd
import sqlite3
import logging
import os

# ════════════════════════════════════════
# 1. CONFIGURACIÓN DE LOGS
# ════════════════════════════════════════

os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/carga.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso ETL Load")

# ════════════════════════════════════════
# 2. CONEXIÓN A BASE DE DATOS
# ════════════════════════════════════════

os.makedirs("database", exist_ok=True)

conn = sqlite3.connect("database/vr_database.db")
cursor = conn.cursor()

logging.info("Conexión a SQLite establecida")

# ════════════════════════════════════════
# 3. CREACIÓN DE TABLAS
# ════════════════════════════════════════

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_gender (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    gender TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_headset (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    headset TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_vr_sessions (
    session_id INTEGER PRIMARY KEY AUTOINCREMENT,
    age INTEGER,
    duration_min REAL,
    motionsickness INTEGER,
    immersionlevel INTEGER,
    vr_experience_score REAL,

    gender_id INTEGER,
    headset_id INTEGER,

    FOREIGN KEY (gender_id)
        REFERENCES dim_gender(id),

    FOREIGN KEY (headset_id)
        REFERENCES dim_headset(id)
)
""")

conn.commit()

logging.info("Tablas creadas correctamente")

# ════════════════════════════════════════
# 4. CARGA DEL CSV VALIDADO
# ════════════════════════════════════════

df = pd.read_csv("/content/data/processed/vr_clean.csv")

logging.info(f"Dataset cargado: {len(df)} registros")

# ════════════════════════════════════════
# 5. INSERTAR DIMENSIONES
# ════════════════════════════════════════

# Géneros únicos
for gender in df["gender_std"].dropna().unique():
    cursor.execute("""
        INSERT OR IGNORE INTO dim_gender (gender)
        VALUES (?)
    """, (gender,))

# Headsets únicos
for headset in df["headset_std"].dropna().unique():
    cursor.execute("""
        INSERT OR IGNORE INTO dim_headset (headset)
        VALUES (?)
    """, (headset,))

conn.commit()

logging.info("Dimensiones cargadas")

# ════════════════════════════════════════
# 6. DICCIONARIOS DE REFERENCIA
# ════════════════════════════════════════

cursor.execute("SELECT id, gender FROM dim_gender")
gender_dict = {row[1]: row[0] for row in cursor.fetchall()}

cursor.execute("SELECT id, headset FROM dim_headset")
headset_dict = {row[1]: row[0] for row in cursor.fetchall()}

# ════════════════════════════════════════
# 7. INSERCIÓN CONTROLADA
# ════════════════════════════════════════

rechazados = []

for index, row in df.iterrows():

    try:

        # Validación referencial
        gender_id = gender_dict.get(row["gender_std"])
        headset_id = headset_dict.get(row["headset_std"])

        if gender_id is None or headset_id is None:
            raise ValueError("Clave foránea inválida")

        cursor.execute("""
            INSERT INTO fact_vr_sessions (
                age,
                duration_min,
                motionsickness,
                immersionlevel,
                vr_experience_score,
                gender_id,
                headset_id
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            int(row["age"]),
            float(row["duration_min"]),
            int(row["motionsickness"]),
            int(row["immersionlevel"]),
            float(row["vr_experience_score"]),
            gender_id,
            headset_id
        ))

    except Exception as e:

        row["error"] = str(e)
        rechazados.append(row)

        logging.error(f"Fila {index} rechazada: {e}")

# ════════════════════════════════════════
# 8. COMMIT FINAL
# ════════════════════════════════════════

conn.commit()

logging.info("Inserción finalizada")

# ════════════════════════════════════════
# 9. GUARDAR RECHAZADOS
# ════════════════════════════════════════

os.makedirs("rejected", exist_ok=True)

rechazados_df = pd.DataFrame(rechazados)

rechazados_df.to_csv(
    "rejected/registros_rechazados.csv",
    index=False
)

logging.info(f"Registros rechazados: {len(rechazados_df)}")

# ════════════════════════════════════════
# 10. CIERRE
# ════════════════════════════════════════

conn.close()

logging.info("Proceso terminado correctamente")

print("✓ Proceso ETL Load completado")
print(f"✓ Registros rechazados: {len(rechazados_df)}")

✓ Proceso ETL Load completado
✓ Registros rechazados: 0
